# 결측치 정밀 대치 실험: 배아 이식 경과일 (구조적 결측 vs 진짜 결측 분리)

**주의**: CatBoost 재튜닝 노트북이 끝난 후에 실행해주세요. 동시에 돌리면
CPU를 나눠 쓰게 돼서 둘 다 느려집니다.

현재 main.py / 기존 노트북들은 `배아 이식 경과일`의 결측치를 전부 -1로 채웁니다.
이건 "이식을 안 한 경우(구조적 결측, 대부분)"와 "이식은 했는데 날짜만 누락된
경우(진짜 결측, Train 약 740건)"를 구분하지 않고 똑같이 처리하는 거예요.

이 노트북은 두 방식을 **동일한 5-Fold split으로 정직하게 비교**합니다:
- **A (기존 방식)**: 전부 -1로 채움
- **B (정밀 대치)**: 구조적 결측은 -1 유지, 진짜 결측만 "실제 이식한 그룹의
  최빈값"(임상적으로 3일 또는 5일)으로 대치

LightGBM만 사용하므로 torch 관련 충돌 걱정은 없습니다.

## 1. 라이브러리 및 설정

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
RANDOM_STATE = 1004

## 2. 공통 전처리 함수

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na_basic(df):
    """기존 방식: 전부 -1 / 해당없음으로 채움"""
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


def fill_na_refined(df):
    """정밀 대치: 배아 이식 경과일의 '진짜 결측치'만 따로 최빈값으로 대치한 뒤,
    나머지(구조적 결측 포함)는 기존 방식과 동일하게 처리"""
    df = df.copy()
    col = "배아 이식 경과일"

    transferred = df["이식된 배아 수"].fillna(0) > 0
    true_missing = transferred & df[col].isna()
    observed_among_transferred = df.loc[transferred & df[col].notna(), col]
    mode_value = observed_among_transferred.mode().iloc[0]

    print(f"[정밀 대치] 실제 이식 그룹의 최빈값: {mode_value}일")
    print(f"[정밀 대치] 진짜 결측치(이식했지만 날짜 모름) 행 수: {true_missing.sum()}건")
    print(f"[정밀 대치] 이 행들만 {mode_value}일로 대치, 나머지(구조적 결측)는 기존과 동일하게 -1 처리")

    df.loc[true_missing, col] = mode_value
    return fill_na_basic(df)


print("전처리 함수 준비 완료")

전처리 함수 준비 완료


## 3. A: 기존 방식 (전부 -1)

In [3]:
df_a = fill_na_basic(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols_a = df_a.select_dtypes(include=["object"]).columns.tolist()

df_a_le = df_a.copy()
for col in cat_cols_a:
    le = LabelEncoder()
    df_a_le[col] = le.fit_transform(df_a_le[col].astype(str))

y = df_a_le[TARGET_COL].values.astype(np.float32)
X_a = df_a_le.drop(columns=[TARGET_COL])

print(f"A(기존) 전처리 완료: {X_a.shape}")

A(기존) 전처리 완료: (256351, 67)


## 4. B: 정밀 대치 방식

In [4]:
df_b = fill_na_refined(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols_b = df_b.select_dtypes(include=["object"]).columns.tolist()

df_b_le = df_b.copy()
for col in cat_cols_b:
    le = LabelEncoder()
    df_b_le[col] = le.fit_transform(df_b_le[col].astype(str))

X_b = df_b_le.drop(columns=[TARGET_COL])

print(f"B(정밀대치) 전처리 완료: {X_b.shape}")

[정밀 대치] 실제 이식 그룹의 최빈값: 5.0일
[정밀 대치] 진짜 결측치(이식했지만 날짜 모름) 행 수: 740건
[정밀 대치] 이 행들만 5.0일로 대치, 나머지(구조적 결측)는 기존과 동일하게 -1 처리
B(정밀대치) 전처리 완료: (256351, 67)


## 5. 동일한 5-Fold split으로 A/B 비교

In [5]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_a = np.zeros(len(y))
oof_b = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_a, y), start=1):
    m_a = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
    m_a.fit(X_a.iloc[tr_idx], y[tr_idx])
    oof_a[val_idx] = m_a.predict_proba(X_a.iloc[val_idx])[:, 1]

    m_b = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
    m_b.fit(X_b.iloc[tr_idx], y[tr_idx])
    oof_b[val_idx] = m_b.predict_proba(X_b.iloc[val_idx])[:, 1]

    auc_a = roc_auc_score(y[val_idx], oof_a[val_idx])
    auc_b = roc_auc_score(y[val_idx], oof_b[val_idx])
    print(f"Fold {fold}/{N_SPLITS}  A(기존): {auc_a:.5f}  |  B(정밀대치): {auc_b:.5f}  |  차이: {auc_b-auc_a:+.5f}")

score_a = roc_auc_score(y, oof_a)
score_b = roc_auc_score(y, oof_b)

print()
print(f"A(기존) 5-Fold 전체 OOF AUC:   {score_a:.5f}")
print(f"B(정밀대치) 5-Fold 전체 OOF AUC: {score_b:.5f}")
print(f"개선: {score_b - score_a:+.5f}")
print()
if score_b - score_a > 0.001:
    print("=> 노이즈 범위를 넘는 의미 있는 개선입니다. main.py에 반영을 검토해보세요.")
else:
    print("=> 노이즈 수준입니다 (예상대로, 영향받는 행이 전체의 0.3%뿐이라 큰 변화는 어려웠을 가능성이 높습니다).")

Fold 1/5  A(기존): 0.74344  |  B(정밀대치): 0.74307  |  차이: -0.00037
Fold 2/5  A(기존): 0.73912  |  B(정밀대치): 0.73926  |  차이: +0.00013
Fold 3/5  A(기존): 0.73744  |  B(정밀대치): 0.73695  |  차이: -0.00050
Fold 4/5  A(기존): 0.73819  |  B(정밀대치): 0.73755  |  차이: -0.00065
Fold 5/5  A(기존): 0.73832  |  B(정밀대치): 0.73861  |  차이: +0.00029

A(기존) 5-Fold 전체 OOF AUC:   0.73925
B(정밀대치) 5-Fold 전체 OOF AUC: 0.73903
개선: -0.00022

=> 노이즈 수준입니다 (예상대로, 영향받는 행이 전체의 0.3%뿐이라 큰 변화는 어려웠을 가능성이 높습니다).
